In [ ]:
# # Initial Imports and Variables
import numpy as np
import torch
import torchvision
import time
import matplotlib.pyplot as plt

import matplotlib
import sns
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# load_dir= "./Data/"
# results_directory="./Results/"
RANDOM_STATE=2025

class_list=['Floor-Bite', 'Floor-Explore', 'Floor-Poke','Stand-Bite', 'Stand-Eat', 'Stand-Explore', 'Stand-Poke']

data_root = r"./data_untouched/" # Contains images of each behavior in separate folders
npz_array_dir="./correlation_npz/all_corrs3.npz"


In [ ]:
%run Utilities.py
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip(line, cell):
    return

# SSIM-Lag Calculation

In [ ]:
import re
import cv2

# image filename pattern, e.g. "AFH1.S1B.RF__5075.jpg"; format: AF{H|L}{rat_id}.S{session_id}{A|B}.RF__{frame_id}.jpg
FILENAME_RE = re.compile(r'(?P<rat>AF[HL]\d+)\.(?P<session>S\d+[AB])\.RF__(?P<frame>\d+)\.jpg', re.IGNORECASE) 


def parse_filename(fname):
    m = FILENAME_RE.match(fname)
    if m is None:
        return None
    return {
        'rat': m.group('rat'),
        'session': m.group('session'),
        'frame': int(m.group('frame'))
    }




def load_behavior_frames(behavior_dir):
    """
    Returns:
        dict[(rat, session)] -> list of (frame_idx, image)
    """
    data = defaultdict(list)

    for fname in os.listdir(behavior_dir):
        info = parse_filename(fname)
        if info is None:
            continue

        path = os.path.join(behavior_dir, fname)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        key = (info['rat'], info['session'])
        data[key].append((info['frame'], img))

    # sort temporally
    for key in data:
        data[key].sort(key=lambda x: x[0])

    return data

# Correlation

In [ ]:
from collections import defaultdict
from skimage.metrics import structural_similarity as ssim


def ssim_similarity(img1, img2):
    return ssim(img1, img2, data_range=255)
    # return 10*np.log10( np.mean((img1)**2) / np.mean((img1-img2)**2) )

def split_into_bouts(frames, gap_threshold=1, min_bout_len=2):
    """
    frames: sorted list of (frame_idx, image) for a single (rat, session).
    A new bout starts when consecutive frames differ by > gap_threshold.

    Returns:
        list of bouts, each bout is a list of (frame_idx, image)
    """
    if not frames:
        return []

    bouts = []
    cur = [frames[0]]

    for (t, img) in frames[1:]:
        prev_t = cur[-1][0]
        if (t - prev_t) > gap_threshold:
            if len(cur) >= min_bout_len:
                bouts.append(cur)
            cur = [(t, img)]
        else:
            cur.append((t, img))

    if len(cur) >= min_bout_len:
        bouts.append(cur)

    return bouts

def autocorr_sum_count_for_bout(bout_frames, similarity_fn, max_lag):
    """
    Bout-level lag similarity stats.
    Uses frame indices, so if there are small gaps inside a bout, pairs whose
    index difference isn't exactly k are skipped.

    Returns:
        sum_sim: dict[k] -> sum of similarities
        count:   dict[k] -> number of valid pairs
    """
    frame_idx = [t for t, _ in bout_frames]
    imgs = [img for _, img in bout_frames]
    n = len(imgs)

    sum_sim = defaultdict(float)
    count = defaultdict(int)

    # For each starting position i, attempt lag steps in the *list*,
    # but only accept if actual frame difference equals k.
    for i in range(n):
        img_i = imgs[i]
        t_i = frame_idx[i]
        max_k_here = min(max_lag, n - 1 - i)
        for k in range(max_k_here + 1):
            j = i + k
            if frame_idx[j] - t_i != k:
                continue
            sum_sim[k] += similarity_fn(img_i, imgs[j])
            count[k] += 1

    return sum_sim, count

In [ ]:

def behavior_autocorrelation_stats_by_bout(
    behavior_dir,
    max_lag=200,
    similarity_fn=ssim_similarity,
    gap_threshold=1,
    min_bout_len=2
):
    sequences = load_behavior_frames(behavior_dir)

    total_sum = np.zeros(max_lag + 1, dtype=np.float64)
    total_count = np.zeros(max_lag + 1, dtype=np.int64)

    n_sequences = 0
    n_bouts = 0

    for (rat, session), frames in sequences.items():
        n_sequences += 1
        bouts = split_into_bouts(frames, gap_threshold=gap_threshold, min_bout_len=min_bout_len)

        for bout in bouts:
            n_bouts += 1
            bout_sum, bout_count = autocorr_sum_count_for_bout(bout, similarity_fn, max_lag)

            for k, s_k in bout_sum.items():
                total_sum[k] += s_k
                total_count[k] += bout_count[k]

    info = {
        "behavior_dir": behavior_dir,
        "n_sequences_(rat,session)": n_sequences,
        "n_bouts": n_bouts,
        "gap_threshold": gap_threshold,
        "min_bout_len": min_bout_len
    }
    return total_sum, total_count, info

def autocorrelation_all_behaviors(
    data_root_dir,
    behavior_names,
    max_lag=200,
    similarity_fn=ssim_similarity,
    gap_threshold=1,
    min_bout_len=2
):
    grand_sum = np.zeros(max_lag + 1, dtype=np.float64)
    grand_count = np.zeros(max_lag + 1, dtype=np.int64)

    per_behavior_corr = {}
    per_behavior_info = {}

    for behavior in behavior_names:
        behavior_dir = os.path.join(data_root_dir, behavior)
        if not os.path.isdir(behavior_dir):
            continue

        total_sum, total_count, info = behavior_autocorrelation_stats_by_bout(
            behavior_dir,
            max_lag=max_lag,
            similarity_fn=similarity_fn,
            gap_threshold=gap_threshold,
            min_bout_len=min_bout_len
        )

        R_behavior = np.zeros(max_lag + 1, dtype=np.float64)
        valid = total_count > 0
        R_behavior[valid] = total_sum[valid] / total_count[valid]

        per_behavior_corr[behavior] = {
            "R": R_behavior,
            "count": total_count
        }

        grand_sum += total_sum
        grand_count += total_count
        per_behavior_info[behavior] = info
        print(f"behavior {behavior} done")

    R_all = np.zeros(max_lag + 1, dtype=np.float64)
    valid = grand_count > 0
    R_all[valid] = grand_sum[valid] / grand_count[valid]

    global_info = {
        "n_behaviors_used": len(per_behavior_info),
        "behaviors_used": sorted(per_behavior_info.keys())
    }
     # per_behavior_corr["StandExplore"]["R"], R_all,per_behavior_corr["StandExplore"]["count"] ,....

    return per_behavior_corr, R_all, grand_count, per_behavior_info, global_info



In [ ]:

behavior_names = class_list  # or a list of folder names


per_behavior_corr, R_all, counts_all, per_b_info, global_info = autocorrelation_all_behaviors(
    data_root, behavior_names, max_lag=600, similarity_fn=ssim_similarity,
    gap_threshold=5, min_bout_len=10)

np.savez_compressed("./correlation_npz/all_corrs3.npz",
                   per_behavior_corr=per_behavior_corr,
                   R_all=R_all, counts_all=counts_all)
print(global_info)


# Load NPZ and Plot

In [ ]:
p=np.load(npz_array_dir,allow_pickle=True)
per_behavior_corr=p["per_behavior_corr"].item()
# R_all=p["R_all"]
# counts_all= p["counts_all"]

In [ ]:
def weighted_average_R_thresholded(per_behavior_corr, class_list, min_count=0):
    # themin count is to avoid counting delays without the "min_count" amount of samples in the average
    # infer maximum lag
    n_lags = max(len(per_behavior_corr[b]["R"])
                 for b in class_list
                 if b in per_behavior_corr)

    weighted_sum = np.zeros(n_lags, dtype=np.float64)
    total_count = np.zeros(n_lags, dtype=np.int64)

    for b in class_list:
        if b not in per_behavior_corr:
            continue

        Rb = per_behavior_corr[b]["R"]
        cb = per_behavior_corr[b]["count"]

        m = min(n_lags, len(Rb), len(cb))
        valid = cb[:m] >= min_count

        weighted_sum[:m][valid] += Rb[:m][valid] * cb[:m][valid]
        total_count[:m][valid] += cb[:m][valid]

    R_all = np.zeros(n_lags, dtype=np.float64)
    valid = total_count > 0
    R_all[valid] = weighted_sum[valid] / total_count[valid]

    return R_all, total_count


R_all_weighted, count_all =  weighted_average_R_thresholded(per_behavior_corr, class_list)


In [ ]:
def draw_xy_plot(
    x, y,
    xlabel,
    ylabel,
    title=None,
    filename=None
):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # color palette
    colors = sns.color_palette("Paired", n_colors=12)

    # font setup (same spirit as your confusion plot)
    font_name = "Times New Roman"
    s = np.array([6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
    s=1.3*np.arange(1,100)
    font_list = 1 * s

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    sns.lineplot(
        x=x,
        y=y,
        ax=ax,
        color=colors[9],
        linewidth=2.5
    )

    ax.set_xlabel(xlabel, fontsize=font_list[12], fontname=font_name, weight="bold")
    ax.set_ylabel(ylabel, fontsize=font_list[12], fontname=font_name, weight="bold")

    if title is not None:
        ax.set_title(title, fontsize=font_list[13], fontname=font_name, weight="bold")

    ax.tick_params(axis="both", labelsize=font_list[11])
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname(font_name)

    # ax.grid(False, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.tick_params(axis="both", direction="in")


    plt.tight_layout()

    if filename is not None:
        plt.savefig(filename, format="pdf", bbox_inches="tight")

    plt.show()



lags = np.arange(len(R_all_weighted))
valid = count_all > 0

draw_xy_plot(np.arange(len(R_all_weighted))[:300],R_all_weighted[:300],
    xlabel="Lag (frames)",
    ylabel="SSIM",
    # title="SSIM–Lag Similarity Curve",
    filename="./Results/Model_Comparison/sfig-ssim.pdf"
)
